# How many SFChron articles link to city-council meeting minutes?

In [84]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from tldextract import extract as extract_domain

banned_domain_list = [
    'sfchronicle',
    '',    
    'sfchron',
    'hearst',
    'myshopify',
    'sfgate',
    'legacy',
    'facebook',
    'twitter',
    'instagram',
    'linkedin',
    'si',
    'theathletic',
    'nbcsports',
    'overthecap',
    'chicagotribune',
    'youtube',
    't',
    'post-gazette',
    'nfl',
    'kerlanjobe',
    'steelersdepot',
    'podfollow',
    'nytimes',
    'yahoo',
    'screenagersmovie',
    'cjr',
    'houstonchronicle',
    'opensea',
    'seasonofsharing',
    'coindesk',
    'theverge',
    'apple',
    'usatoday',
    'voicemap',
    'eventbrite',
]
banned_text_list = [
    'Careers',
    'Reprints & Permissions',
    'Go Back',
    'Send Feedback',
    'Cancel',
    'HNP',
]


def filter_urls(url_chunk):
    url = url_chunk['href']
    text = url_chunk['text'].strip()
    if url is None:
        return False
    if text in banned_text_list:
        return False
    if extract_domain(url).domain in banned_domain_list:
        return False
    return True

In [86]:
import jsonlines

In [ ]:
with open('../data/sfchron-fetched-articles-with-html.jsonl') as f:
    sfchron_htmls = list(jsonlines.Reader(f))

sfchron_htmls_df = pd.DataFrame(sfchron_htmls)

In [99]:
all_urls = []
for _, (article_url, html) in tqdm(sfchron_htmls_df[['article_url', 'article_html']].iterrows(), total=len(sfchron_htmls_df)):
    soup = BeautifulSoup(html)
    a_s = soup.find_all('a')
    a_s = list(map(lambda x: {
        'href': x.attrs.get('href'),
        'text': x.get_text(),
        'article_url': article_url,
    }, a_s))
    filtered_urls = list(filter(filter_urls, a_s))
    all_urls += filtered_urls

all_urls_df = pd.DataFrame(all_urls)

  0%|          | 0/11331 [00:00<?, ?it/s]

In [185]:
legistar_url, _, sfchron_url = (
    all_urls_df
    .loc[lambda df: df['href'].str.contains('legistar', regex=False)]
    .assign(article_url=lambda df: 'https://www.sfchronicle.com' + df['article_url'].str.split(')').str.get(-1))
    .iloc[26]
    .values
)

In [159]:
import pyperclip
pyperclip.copy(legistar_url)

In [186]:
pyperclip.copy(sfchron_url)

In [144]:
sfchron_url

'https://www.sfchronicle.com/bayarea/article/sf-proposal-targets-bike-chop-shops-as-12247776.php'

In [ ]:
(all_urls_df['href']
 .apply(lambda x: extract_domain(x).domain)
 .value_counts()
 .index.tolist()
)

(all_urls_df['href']
 .apply(lambda x: extract_domain(x).suffix).value_counts()
)

(all_urls_df['href']
 .loc[lambda s: s.apply(lambda x: extract_domain(x).domain) == 'ca']
)

(all_urls_df['href']
 .loc[lambda s: s.apply(lambda x: extract_domain(x).domain) == 'legistar']
 .values
)

legistar_df = pd.read_csv('../bin/aw-legistar-scraper-selenium/legistar_scraper/output-2013/mountain-view.csv')
legistar_df['url'].iloc[0]

# Hyperlink Analysis

In [ ]:
open('/Users/spangher/Downloads/example_arcs').read()
open('/Users/spangher/Downloads/example_index').read().split('\n')
domain_index = pd.read_csv('../data/hyperlink-graph-data/index.pld', sep='\t', header=None)
domain_index.columns = ['domain', 'id']
domain_index.loc[lambda df: df['domain'] == 'legistar.com']

In [11]:
import glob
import pandas as pd 
page_level_data = sorted(glob.glob('../data/hyperlink-graph-data/index-00*'))

In [12]:
page_index = pd.read_csv(page_level_data[0], sep='\t', header=None)
page_index.columns = ['domain', 'id']

In [15]:
page_index[0].iloc[-1]

'http://barrowmediacenter.wordpress.com/2009/10/29/2009-storybook-parade/storybook-parade-and-jittery-joes-134/'

In [47]:
import urllib
def unquote_recursive(url):
    prev_url = url
    unquoted_url = urllib.parse.unquote(prev_url)
    while unquoted_url != prev_url:
        prev_url = unquoted_url
        unquoted_url = urllib.parse.unquote(prev_url)
    return unquoted_url

In [52]:
from tqdm.auto import tqdm

In [54]:
tqdm.pandas()
page_index = page_index.assign(domain=lambda df: df['domain'].progress_apply(unquote_recursive))

  0%|          | 0/32835720 [00:00<?, ?it/s]

In [58]:
legistar_links = page_index.loc[lambda df: df['domain'].str.contains('legistar')]

In [61]:
legistar_links.iloc[1].values

array(['http://a2gov.legistar.com/DepartmentDetail.aspx?ID=12515&GUID=AED99AEC-1AA7-4FFF-9915-66917C506AFF',
       3211743], dtype=object)

In [62]:
legistar_links['id']

,domain,id
3211742,http://a2gov.legistar.com/Calendar.aspx,3211742
3211743,http://a2gov.legistar.com/DepartmentDetail.asp...,3211743
3211744,http://a2gov.legistar.com/DepartmentDetail.asp...,3211744
3211745,http://a2gov.legistar.com/DepartmentDetail.asp...,3211745
3211746,http://a2gov.legistar.com/DepartmentDetail.asp...,3211746
3211747,http://a2gov.legistar.com/DepartmentDetail.asp...,3211747
3211748,http://a2gov.legistar.com/DepartmentDetail.asp...,3211748
3211749,http://a2gov.legistar.com/DepartmentDetail.asp...,3211749
3211750,http://a2gov.legistar.com/DepartmentDetail.asp...,3211750
3211751,http://a2gov.legistar.com/DepartmentDetail.asp...,3211751


In [69]:
import gzip
arc_graph = pd.read_csv('../data/hyperlink-graph-data/part-r-00000.gz', compression='gzip', sep='\t', header=None) 

In [71]:
arc_graph.loc[lambda df: df[0].isin(legistar_links['id'])]

,0,1
64354788,3211742,3207115
64354789,3211742,3207305
64354790,3211742,3211742
64354791,3211742,3219900
64354792,3211742,3234478
...,...,...
64356019,3211766,154783160
64356020,3211766,164947986
64356021,3211766,387452309
64356022,3211766,469663297


In [72]:
arc_graph.loc[lambda df: df[1].isin(legistar_links['id'])]

,0,1
64174812,3207901,3211760
64216856,3208771,3211762
64273006,3209979,3211760
64347571,3211600,3211766
64349532,3211638,3211745
...,...,...
75423797,3444233,3211761
75446423,3447805,3211762
75635791,3476382,3211751
75665567,3480513,3211748


In [81]:
page_index.loc[lambda df: df['domain'].str.contains('sfbos.org', regex=False)]

,domain,id


In [190]:
pd.Series(t.split('\n')).value_counts()

Steven Melendez    55
Alex Spangher      18
Serdar Tumgoren     1
dtype: int64

In [ ]:
# 10,000 news articles, 500 links 
# highest position on the homepage of the sfchron articles that had these links 

# get a set of a